# 서울시 따릉이 데이터 전처리 (ISLP 모델링 목적 - 원본 이력 보존형)

이 노트북은 서울시 공공자전거(따릉이) 데이터와 기상/미세먼지 데이터를 결합하여 ISLP 다중선형회귀 및 교호작용 분석에 활용할 수 있도록 전처리하는 파이프라인입니다.

**[오류 방지 및 서울 108번 데이터 엄격 필터링 적용]**

## 1. 라이브러리 및 경로 설정

In [26]:
import os
import glob
import pandas as pd
import numpy as np
import traceback

BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, 'data')

WEATHER_DIR = os.path.join(DATA_DIR, '기상정보(03.31_05.21)')
BIKE_DIR = os.path.join(DATA_DIR, '따릉이(03,04,05.01_05.15)')
MASTER_DIR = os.path.join(DATA_DIR, '따릉이_마스터정보(대여소_id)')
DUST_DIR = os.path.join(DATA_DIR, '미세먼지 정보(03.31_05.21)')

START_DATE = '2026-03-01'
END_DATE = '2026-05-15'

def find_encoding(file_path):
    for enc in ['utf-8', 'cp949', 'euc-kr']:
        try:
            pd.read_csv(file_path, encoding=enc, nrows=1)
            return enc
        except UnicodeDecodeError:
            continue
    return 'cp949'

## 2. 마스터 데이터 기반 타겟 대여소 추출

In [27]:
try:
    def load_master_data():
        master_files = glob.glob(os.path.join(MASTER_DIR, '*.csv'))
        if not master_files:
            raise FileNotFoundError("마스터 데이터 폴더에 CSV 파일이 없습니다.")
        
        master_file = master_files[0]
        enc = find_encoding(master_file)
        master_df = pd.read_csv(master_file, encoding=enc)
        
        station_name_col = next((col for col in master_df.columns if '명' in col or '이름' in col), master_df.columns[2])
        station_id_col = next((col for col in master_df.columns if 'ID' in col.upper() or '번호' in col), master_df.columns[0])
        
        office_keywords = ['여의도역', '강남역', '광화문역', '을지로입구역', '시청역', '선릉역', '삼성역', '종각역', '가산디지털단지역', '판교역', '공덕역']
        leisure_keywords = ['여의나루역', '뚝섬유원지역', '서울숲역', '반포한강공원', '망원한강공원', '올림픽공원', '월드컵경기장', '어린이대공원', '노들섬', '석촌호수']
        university_keywords = ['공릉', '서울과학기술대학교']

        def assign_station_type(name):
            name = str(name)
            for kw in office_keywords:
                if kw in name: return 'Office'
            for kw in leisure_keywords:
                if kw in name: return 'Leisure'
            for kw in university_keywords:
                if kw in name: return 'University'
            return None

        master_df['station_type'] = master_df[station_name_col].apply(assign_station_type)
        target_master_df = master_df.dropna(subset=['station_type']).copy()
        
        target_master_df = target_master_df[[station_id_col, station_name_col, 'station_type']]
        target_master_df.columns = ['station_id', 'station_name', 'station_type']
        target_master_df = target_master_df.drop_duplicates(subset=['station_id'])
        
        return target_master_df

    target_master_df = load_master_data()
    print(f"{len(target_master_df)}개의 타겟 대여소 식별 완료.")
    display(target_master_df.head())
except Exception as e:
    print("마스터 데이터 로드 중 오류 발생:")
    traceback.print_exc()

31개의 타겟 대여소 식별 완료.


,station_id,station_name,station_type
10,ST-99,뚝섬유원지역 1번출구 앞,Leisure
48,ST-955,삼성역 5~6번 출구,Office
143,ST-87,홈플러스 월드컵경기장점,Leisure
214,ST-805,삼성역 3번 출구,Office
219,ST-800,삼성역 7번출구,Office


## 3. 대규모 따릉이 원본 이력 추출

In [28]:
try:
    def load_and_filter_raw_bike_data(folder_path, target_master_df):
        all_files = glob.glob(os.path.join(folder_path, '**', '*.csv'), recursive=True)
        df_list = []
        
        target_station_ids = target_master_df['station_id'].unique()

        for file in all_files:
            enc = find_encoding(file)
            df = pd.read_csv(file, encoding=enc)
            
            start_id_cols = [c for c in df.columns if '시작' in c and 'ID' in c.upper()]
            if not start_id_cols:
                start_id_cols = [c for c in df.columns if '대여소_ID' in c.upper() or '대여대여소' in c]
                
            if not start_id_cols: continue
                
            join_col = start_id_cols[0]
            
            filtered_df = df[df[join_col].isin(target_station_ids)].copy()
            if filtered_df.empty: continue
                
            filtered_df = pd.merge(filtered_df, target_master_df[['station_id', 'station_type']], left_on=join_col, right_on='station_id', how='inner')
            filtered_df.drop(columns=['station_id'], inplace=True)
            
            date_col = next((c for c in filtered_df.columns if '날짜' in c or '일자' in c), filtered_df.columns[0])
            filtered_df['date'] = pd.to_datetime(filtered_df[date_col].astype(str).str.replace('-', ''), errors='coerce').dt.normalize()
            filtered_df = filtered_df.dropna(subset=['date'])
            
            df_list.append(filtered_df)
                
        if not df_list:
            return pd.DataFrame()
            
        merged_bike_df = pd.concat(df_list, ignore_index=True)
        return merged_bike_df

    bike_raw_df = load_and_filter_raw_bike_data(BIKE_DIR, target_master_df)
    if bike_raw_df.empty:
        print("따릉이 데이터가 비어있습니다. 경로를 확인해주세요.")
    else:
        print(f"필터링된 원본 데이터 병합 완료: 총 {len(bike_raw_df):,} 건")
        display(bike_raw_df.head())
except Exception as e:
    print("따릉이 이력 로드 중 오류 발생:")
    traceback.print_exc()

필터링된 원본 데이터 병합 완료: 총 232,648 건


,기준_날짜,집계_기준,기준_시간대,시작_대여소_ID,시작_대여소명,종료_대여소_ID,종료_대여소명,전체_건수,전체_이용_분,전체_이용_거리,station_type,date
0,20260508,출발시간,1540,ST-1586,잠실6동_001_1,ST-811,역삼1동_009_1,4,195.0,30067.0,Leisure,2026-05-08
1,20260508,출발시간,1400,ST-73,여의동_010_1,ST-3401,여의동_006_9,1,91.0,7210.0,Leisure,2026-05-08
2,20260508,출발시간,1740,ST-99,자양3동_036_1,ST-1335,면목2동_026_1,1,62.0,12552.0,Leisure,2026-05-08
3,20260508,출발시간,1925,ST-1720,오륜동_001_4,ST-505,성내3동_012_1,1,13.0,2080.0,Leisure,2026-05-08
4,20260508,출발시간,1655,ST-73,여의동_010_1,ST-2491,가양1동_001_3,1,52.0,11009.0,Leisure,2026-05-08


## 4. 기상 및 미세먼지 환경 데이터 보간 (지점번호 108번 강력 필터링)

In [29]:
try:
    def load_environment_data(folder_path, value_col_hints, output_col_name):
        all_files = glob.glob(os.path.join(folder_path, '**', '*.csv'), recursive=True)
        df_list = []
        
        for file in all_files:
            enc = find_encoding(file)
            df = pd.read_csv(file, encoding=enc)
            
            # 지점코드 확인 및 108 필터링
            station_col = next((c for c in df.columns if '지점' in c), None)
            if not station_col:
                station_col = df.columns[0] # 통상 첫 컬럼이 지점코드
                
            station_ids = pd.to_numeric(df[station_col], errors='coerce')
            df = df[station_ids == 108].copy() # 108번 서울 이외 타 지점 완벽 배제
            if df.empty:
                continue
            
            date_col = next((c for c in df.columns if '일시' in c or '일자' in c or '날짜' in c), None)
            if not date_col:
                date_col = df.columns[2] # 3번째 컬럼
                
            val_cols = []
            for hint in value_col_hints:
                found = next((c for c in df.columns if hint in c), None)
                if found: val_cols.append(found)
                
            if val_cols:
                sub_df = df[[date_col] + val_cols].copy()
                sub_df['date'] = pd.to_datetime(sub_df[date_col], errors='coerce').dt.normalize()
                sub_df = sub_df.dropna(subset=['date'])
                sub_df = sub_df.drop(columns=[date_col])
                df_list.append(sub_df)
                
        if not df_list:
            return pd.DataFrame()
            
        merged_env = pd.concat(df_list, ignore_index=True)
        merged_env = merged_env.groupby('date').mean().reset_index()
        
        rename_dict = {}
        for hint, out_name in zip(value_col_hints, output_col_name):
            found = next((c for c in merged_env.columns if hint in c), None)
            if found:
                rename_dict[found] = out_name
                
        merged_env = merged_env.rename(columns=rename_dict)
        return merged_env

    weather_df = load_environment_data(WEATHER_DIR, ['기온', '강수'], ['temp', 'precip'])
    dust_df = load_environment_data(DUST_DIR, ['미세먼지', 'PM10'], ['pm10', 'pm10'])

    if not weather_df.empty and not dust_df.empty:
        env_df = pd.merge(weather_df, dust_df, on='date', how='outer')
    else:
        env_df = weather_df if not weather_df.empty else dust_df

    if env_df.empty:
        print("환경 데이터를 로드하지 못했습니다. 파일에 108 지점 데이터가 있는지 확인하세요.")
    else:
        env_df.set_index('date', inplace=True)
        env_df.sort_index(inplace=True)

        target_dates = pd.date_range(start=START_DATE, end=END_DATE)
        env_df = env_df.reindex(target_dates)

        if 'precip' in env_df.columns:
            env_df['precip'] = env_df['precip'].fillna(0)
        if 'temp' in env_df.columns:
            env_df['temp'] = env_df['temp'].interpolate(method='time').bfill().ffill()
        if 'pm10' in env_df.columns:
            env_df['pm10'] = env_df['pm10'].interpolate(method='time').bfill().ffill()
            
        env_df.reset_index(inplace=True)
        env_df.rename(columns={'index': 'date'}, inplace=True)
        print("환경 데이터 보간 처리 완료 (지점: 108 서울 전용).")
        display(env_df.head())
except Exception as e:
    print("환경 데이터 로드 중 오류 발생:")
    traceback.print_exc()

환경 데이터 보간 처리 완료 (지점: 108 서울 전용).


,date,temp,precip,pm10
0,2026-03-01,9.1,0.53,22.0
1,2026-03-02,4.9,17.25,24.0
2,2026-03-03,5.9,4.92,26.0
3,2026-03-04,6.0,0.00,28.0
4,2026-03-05,6.0,6.50,53.0


## 5. 최종 결합 및 저장

In [31]:
try:
    if not bike_raw_df.empty and not env_df.empty:
        final_df = pd.merge(bike_raw_df, env_df, on='date', how='left')

        mask = (final_df['date'] >= pd.to_datetime(START_DATE)) & (final_df['date'] <= pd.to_datetime(END_DATE))
        final_df = final_df.loc[mask].copy()

        final_df['is_weekend'] = final_df['date'].dt.dayofweek.isin([5, 6]).astype(int)
        final_df['station_type'] = final_df['station_type'].astype('category')
        final_df['is_weekend'] = final_df['is_weekend'].astype('category')

        if 'date' in final_df.columns:
            cols = final_df.columns.tolist()
            cols.insert(0, cols.pop(cols.index('date')))
            final_df = final_df[cols]
        final_df = final_df.sort_values(by='date').reset_index(drop=True)

        print("[최종 데이터프레임 구조 확인]")
        final_df.info()

        output_path = os.path.join(BASE_DIR, 'preprocessed_data_for_islp.csv')
        final_df.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"\n{START_DATE} ~ {END_DATE} 구간 대여 이력 전처리 성공! '{output_path}' 에 저장되었습니다.")
        display(final_df.head())
    else:
        print("따릉이 또는 환경 데이터가 비어 있어 최종 병합을 수행할 수 없습니다.")
except Exception as e:
    print("최종 결합 및 저장 중 오류 발생:")
    traceback.print_exc()

[최종 데이터프레임 구조 확인]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 232648 entries, 0 to 232647
Data columns (total 16 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   date          232648 non-null  datetime64[ns]
 1   기준_날짜         232648 non-null  int64         
 2   집계_기준         232648 non-null  object        
 3   기준_시간대        232648 non-null  int64         
 4   시작_대여소_ID     232648 non-null  object        
 5   시작_대여소명       227993 non-null  object        
 6   종료_대여소_ID     232648 non-null  object        
 7   종료_대여소명       226740 non-null  object        
 8   전체_건수         232648 non-null  int64         
 9   전체_이용_분       225968 non-null  float64       
 10  전체_이용_거리      225968 non-null  float64       
 11  station_type  232648 non-null  category      
 12  temp          232648 non-null  float64       
 13  precip        232648 non-null  float64       
 14  pm10          232648 non-null  float64       
 15 

,date,기준_날짜,집계_기준,기준_시간대,시작_대여소_ID,시작_대여소명,종료_대여소_ID,종료_대여소명,전체_건수,전체_이용_분,전체_이용_거리,station_type,temp,precip,pm10,is_weekend
0,2026-03-01,20260301,도착시간,2355,ST-1590,잠실3동_053_3,ST-891,잠실6동_001_7,1,4.0,1017.0,Leisure,9.1,0.53,22.0,1
1,2026-03-01,20260301,도착시간,1330,ST-87,성산2동_001_2,ST-342,성산1동_022_1,2,19.0,2594.0,Leisure,9.1,0.53,22.0,1
2,2026-03-01,20260301,도착시간,1330,ST-87,성산2동_001_2,ST-344,성산2동_076_2,1,3.0,533.0,Leisure,9.1,0.53,22.0,1
3,2026-03-01,20260301,출발시간,1335,ST-1123,공릉1동_049_1,ST-2799,도봉2동_001_4,2,66.0,12770.0,University,9.1,0.53,22.0,1
4,2026-03-01,20260301,출발시간,1335,ST-1123,공릉1동_049_1,ST-735,공릉1동_016_1,1,5.0,506.0,University,9.1,0.53,22.0,1
